<a href="https://colab.research.google.com/github/Priya-Kumari-Chourasia/ml_algo/blob/main/bagging_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split


In [2]:
x,y = make_classification(n_samples=10000,n_features=10,n_informative=3)

In [3]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [4]:
x_train

array([[-2.09757227, -1.68268129, -2.74620677, ...,  0.57609949,
        -1.84620571,  0.37624821],
       [-0.38835032,  0.20358234, -0.7996253 , ..., -0.19036939,
        -1.17946075,  1.95928158],
       [-1.76377008, -0.7314529 , -0.52635918, ...,  2.36961191,
         0.10632097,  0.09715017],
       ...,
       [ 0.24558092, -0.80888985, -2.08912411, ..., -0.52850145,
        -1.95275929, -0.09256786],
       [-1.73754241, -1.24296065, -2.39501652, ..., -0.0651507 ,
        -1.81923884,  0.25521232],
       [-0.66890392, -0.81710582, -1.21908868, ..., -0.81427362,
        -0.77355284,  0.67453425]])

In [5]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(x_train,y_train)
y_pred = dt.predict(x_test)

print("Decision Tree accuracy",accuracy_score(y_test,y_pred))

Decision Tree accuracy 0.874


## Bagging

In [7]:
bag = BaggingClassifier(
    estimator = DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=True,
    random_state=42

)

In [8]:
bag.fit(x_train,y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(), max_samples=0.25,
                  n_estimators=500, random_state=42)

In [9]:
y_pred = bag.predict(x_test)

In [10]:
accuracy_score(y_test,y_pred)

0.919

## Bagging using SVM

In [11]:
bag = BaggingClassifier(
    estimator = SVC(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=True,
    random_state=42
)

In [12]:
bag.fit(x_train,y_train)


BaggingClassifier(estimator=SVC(), max_samples=0.25, n_estimators=500,
                  random_state=42)

In [13]:
y_pred = bag.predict(x_test)

In [15]:
print("Bagging using SVM",accuracy_score(y_test,y_pred))

Bagging using SVM 0.904


## Pasting

In [16]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=False,
    random_state=42,
    verbose=1,
    n_jobs=-1
)

In [17]:
bag.fit(x_train,y_train)
y_pred = bag.predict(x_test)
print("Pasting Classifier",accuracy_score(y_test,y_pred))

[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   2 out of   2 | elapsed:   18.1s finished
[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.


Pasting Classifier 0.925


[Parallel(n_jobs=2)]: Done   2 out of   2 | elapsed:    0.6s finished


## Random Subspaces

In [18]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=1.0,
    bootstrap=False,
    max_features=0.5,
    bootstrap_features=True,
    random_state=42
)
bag.fit(x_train,y_train)
y_pred = bag.predict(x_test)
print("Random Subspaces Classifier",accuracy_score(y_test,y_pred))

Random Subspaces Classifier 0.916


## Random Patches

In [19]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=True,
    max_features=0.5,
    bootstrap_features=True,
    random_state=42
)
bag.fit(x_train,y_train)
y_pred = bag.predict(x_test)
print("Random Patches Classifier",accuracy_score(y_test,y_pred))

Random Patches Classifier 0.9165


## OOB Score (out of bag sample)

In [20]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=True,
    oob_score=True,
    random_state=42
)

In [22]:
bag.fit(x_train,y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(), max_samples=0.25,
                  n_estimators=500, oob_score=True, random_state=42)

In [23]:
bag.oob_score_

0.918125

In [24]:
y_pred = bag.predict(x_test)
print("Accuracy",accuracy_score(y_test,y_pred))

Accuracy 0.919


## Bagging Tips


*   Bagging generally gives better results than Pasting
*   Good results come around the 25% to 50% row sampling mark

*   Random patches and subspaces should be used while dealing with high dimensional data
*   To find the correct hyperparameter values we can do GridSearchCV/RandomSearchCV





## Applying GridSearchCV

In [25]:
from sklearn.model_selection import GridSearchCV

In [26]:
parameters = {
    'n_estimators': [50,100,500],
    'max_samples': [0.1,0.4,0.7,1.0],
    'bootstrap' : [True,False],
    'max_features' : [0.1,0.4,0.7,1.0]
    }


In [27]:
search = GridSearchCV(BaggingClassifier(), parameters, cv=5)


In [29]:
search.fit(x_train,y_train)

GridSearchCV(cv=5, estimator=BaggingClassifier(),
             param_grid={'bootstrap': [True, False],
                         'max_features': [0.1, 0.4, 0.7, 1.0],
                         'max_samples': [0.1, 0.4, 0.7, 1.0],
                         'n_estimators': [50, 100, 500]})

In [30]:
search.best_params_
search.best_score_

np.float64(0.92325)

In [31]:
search.best_params_

{'bootstrap': False,
 'max_features': 0.7,
 'max_samples': 0.7,
 'n_estimators': 500}